# Handling Imbalanced Datasets

**Class Imbalance** occurs when the classes in a classification dataset are not represented equally. This is extremely common in real-world applications such as credit card fraud detection (99.9% legitimate, 0.1% fraud), medical diagnosis (rare diseases), and network intrusion detection.

In these detailed notes, we will cover:
1. **The Fatal Flaw of Accuracy:** Why accuracy is misleading on skewed data.
2. **Alternative Evaluation Metrics:** Precision, Recall, F1-Score, G-Mean, and ROC-AUC vs. PR-AUC.
3. **Resampling Techniques:** Random Under-Sampling, Random Over-Sampling, SMOTE, and ADASYN.
4. **Algorithmic Methods:** Cost-sensitive learning (`class_weight='balanced'`).
5. **Practical Python Demos:**
   - **Demo A:** Custom scratch implementation of `RandomUnderSampler` and a basic `SMOTE` (Synthetic Minority Over-sampling Technique) from scratch.
   - **Demo B:** Generating a synthetic imbalanced dataset and visualizing the decision boundaries under different sampling methods.
   - **Demo C:** Evaluation comparison showing how ROC-AUC hides poor performance on the minority class while PR-AUC exposes it.
   - **Summary Checklist** for handling imbalanced data.

## 1. The Fatal Flaw of Accuracy & Alternative Metrics

### Why Accuracy Fails
Suppose a credit card dataset has $100,000$ transactions, where $99,900$ are legitimate and $100$ are fraudulent ($0.1\%$ imbalance).
- If we write a naive model that predicts **"Legitimate"** for every single transaction, the accuracy is:
  $$\text{Accuracy} = \frac{99,900}{100,000} = 99.9\%$$
- Despite the $99.9\%$ accuracy, this model is completely useless because it detects **zero** fraud cases. Thus, **accuracy is not a reliable metric for imbalanced datasets.**

### The Confusion Matrix & Advanced Metrics
To evaluate imbalanced classifiers, we look at the counts of True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN):

| Metric | Formula | What it measures |
| :--- | :--- | :--- |
| **Recall (Sensitivity)** | $$\frac{\text{TP}}{\text{TP} + \text{FN}}$$ | Out of all actual positives, how many did we find? (Crucial for medical diagnosis) |
| **Precision (PPV)** | $$\frac{\text{TP}}{\text{TP} + \text{FP}}$$ | Out of all predicted positives, how many are actually positive? (Crucial for spam filter spam folder) |
| **F1-Score** | $$2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$ | Harmonic mean of Precision and Recall. Balances both metrics. |
| **Geometric Mean (G-Mean)** | $$\sqrt{\text{TPR} \cdot \text{TNR}}$$ | Balances performance on both majority and minority classes. |
| **Balanced Accuracy** | $$\frac{\text{Sensitivity} + \text{Specificity}}{2}$$ | Average of recall obtained on each class. |

---

### ROC-AUC vs. PR-AUC
- **ROC Curve (Receiver Operating Characteristic):** Plots True Positive Rate (TPR) vs. False Positive Rate (FPR).
  - *FPR* = $\frac{\text{FP}}{\text{TN} + \text{FP}}$. In highly imbalanced data, TN is very large, so FPR remains small even if FP is large. This makes the ROC curve look overly optimistic.
- **PR Curve (Precision-Recall):** Plots Precision vs. Recall.
  - Precision incorporates FP directly against TP rather than the massive TN pool. Therefore, the PR curve is far more sensitive to false positives in the minority class and is the preferred visualization for highly imbalanced classifications.

## 2. Resampling & Algorithmic Techniques

Resampling modifies the training dataset distribution to balance the classes before feeding them into an ML algorithm.

### A. Under-sampling (Down-sampling)
Reduces the size of the majority class to match the minority class.
- **Random Under-sampling (RUS):** Randomly removes majority class samples.
  - *Pro:* Reduces training time and memory footprint.
  - *Con:* Discards valuable information that could help define the decision boundary.
- **Tomek Links:** Identifies pairs of opposite-class samples that are nearest neighbors to each other. Removing the majority class sample in the pair clears the overlap near the decision boundary.

### B. Over-sampling (Up-sampling)
Increases the representation of the minority class.
- **Random Over-sampling (ROS):** Randomly duplicates minority class samples.
  - *Pro:* Retains all information.
  - *Con:* Leads to severe overfitting as the model learns exact copies of minority samples.
- **SMOTE (Synthetic Minority Over-sampling Technique):** Synthesizes new, unique samples rather than duplicating them.
  - *How SMOTE works:* 
    1. For each minority sample $x$, find its $k$-nearest neighbors in the minority class.
    2. Choose one neighbor $x_{\text{neigh}}$ at random.
    3. Create a synthetic sample along the line segment joining the two points:
       $$x_{\text{new}} = x + \lambda \cdot (x_{\text{neigh}} - x) \quad \text{where } \lambda \sim \text{Uniform}(0, 1)$$
    4. Repeat until desired balance is achieved.
- **ADASYN (Adaptive Synthetic):** Similar to SMOTE, but generates more synthetic data for minority samples that are harder to learn (i.e., those surrounded by majority class samples).

### C. Cost-Sensitive Learning
Instead of changing the dataset, modify the algorithm's loss function to penalize misclassifications of the minority class more severely. In Scikit-Learn, this is typically configured using `class_weight='balanced'` which assigns weights inversely proportional to class frequencies:
$$w_j = \frac{N}{\text{n\_classes} \cdot n_j}$$

## 3. Code Setup & Imports

Let's prepare the Python environment. We will use standard scientific libraries and check for the presence of Scikit-Learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score, precision_score, 
    recall_score, f1_score, roc_curve, roc_auc_score, precision_recall_curve, auc
)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
np.random.seed(42)

print("Libraries imported successfully!")

## 4. Demo A: Scratch Implementations of RUS and SMOTE

To understand resampling, we will build custom scratch classes:
1. `RandomUnderSamplerScratch`: Randomly selects a subset of the majority class.
2. `SMOTEScratch`: Implements the vector interpolation method to synthesize minority class points.

In [ ]:
class RandomUnderSamplerScratch:
    def __init__(self, random_state=42):
        self.random_state = random_state

    def fit_resample(self, X, y):
        np.random.seed(self.random_state)
        
        # Identify indices of majority and minority classes
        classes, counts = np.unique(y, return_counts=True)
        minority_class = classes[np.argmin(counts)]
        majority_class = classes[np.argmax(counts)]
        
        minority_idx = np.where(y == minority_class)[0]
        majority_idx = np.where(y == majority_class)[0]
        
        n_minority = len(minority_idx)
        
        # Randomly sample from majority indices to match minority count
        undersampled_majority_idx = np.random.choice(majority_idx, size=n_minority, replace=False)
        
        # Combine indices
        combined_idx = np.concatenate([minority_idx, undersampled_majority_idx])
        np.random.shuffle(combined_idx)
        
        return X[combined_idx], y[combined_idx]


class SMOTEScratch:
    def __init__(self, k_neighbors=5, random_state=42):
        self.k_neighbors = k_neighbors
        self.random_state = random_state

    def fit_resample(self, X, y):
        np.random.seed(self.random_state)
        
        # Identify minority class
        classes, counts = np.unique(y, return_counts=True)
        minority_class = classes[np.argmin(counts)]
        majority_class = classes[np.argmax(counts)]
        
        minority_idx = np.where(y == minority_class)[0]
        majority_idx = np.where(y == majority_class)[0]
        
        X_min = X[minority_idx]
        n_minority = len(X_min)
        n_majority = len(majority_idx)
        n_synthetic_needed = n_majority - n_minority
        
        if n_synthetic_needed <= 0:
            return X, y
            
        # Fit NearestNeighbors on the minority class points
        nn = NearestNeighbors(n_neighbors=self.k_neighbors + 1)
        nn.fit(X_min)
        
        synthetic_samples = []
        for _ in range(n_synthetic_needed):
            # Select a random minority point index
            idx = np.random.randint(0, n_minority)
            point = X_min[idx]
            
            # Find its k-neighbors indices (excluding itself at distance index 0)
            _, neigh_indices = nn.kneighbors(point.reshape(1, -1))
            neigh_indices = neigh_indices[0][1:] # Exclude self
            
            # Pick a random neighbor
            rand_neigh_idx = np.random.choice(neigh_indices)
            neighbor = X_min[rand_neigh_idx]
            
            # Synthesize point: point + lambda * (neighbor - point)
            lam = np.random.rand()
            synthetic_point = point + lam * (neighbor - point)
            synthetic_samples.append(synthetic_point)
            
        # Stack synthetic points back with original data
        X_synthetic = np.array(synthetic_samples)
        y_synthetic = np.full(n_synthetic_needed, minority_class)
        
        X_resampled = np.vstack([X, X_synthetic])
        y_resampled = np.concatenate([y, y_synthetic])
        
        # Shuffle resampled dataset
        shuffle_idx = np.random.permutation(len(y_resampled))
        return X_resampled[shuffle_idx], y_resampled[shuffle_idx]

print("Scratch resampling classes defined successfully!")

## 5. Demo B: Visually Comparing Resampling Decision Boundaries

Let's generate a synthetic 2D binary dataset with a **95% / 5% class split** to observe how training boundaries change when we fit Logistic Regression under different conditions.

In [ ]:
# Generate 2D synthetic imbalanced data
X_raw, y_raw = make_classification(
    n_samples=600, n_features=2, n_redundant=0, n_clusters_per_class=1,
    weights=[0.95, 0.05], class_sep=1.0, random_state=42
)

# Apply scratch samplers
rus = RandomUnderSamplerScratch(random_state=42)
X_rus, y_rus = rus.fit_resample(X_raw, y_raw)

smote = SMOTEScratch(k_neighbors=5, random_state=42)
X_smote, y_smote = smote.fit_resample(X_raw, y_raw)

# Visualize distributions after resampling
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

def plot_scatter(ax, X, y, title):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], color='#3498db', alpha=0.6, label='Majority (Class 0)', edgecolors='w')
    ax.scatter(X[y == 1, 0], X[y == 1, 1], color='#e74c3c', alpha=0.9, label='Minority (Class 1)', edgecolors='k')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()

plot_scatter(axes[0], X_raw, y_raw, f"Original Imbalanced Data\nClass 0: {np.sum(y_raw==0)} | Class 1: {np.sum(y_raw==1)}")
plot_scatter(axes[1], X_rus, y_rus, f"Random Under-sampled (RUS)\nClass 0: {np.sum(y_rus==0)} | Class 1: {np.sum(y_rus==1)}")
plot_scatter(axes[2], X_smote, y_smote, f"SMOTE Resampled\nClass 0: {np.sum(y_smote==0)} | Class 1: {np.sum(y_smote==1)}")

plt.tight_layout()
plt.show()

### Decision Boundary Visualization
Now we plot the decision boundaries of Logistic Regression classifiers fitted on the unscaled raw data vs. the balanced options.

In [ ]:
# Train Logistic Regression classifiers
models = {
    "Unbalanced Baseline": (LogisticRegression(), X_raw, y_raw),
    "Class Weighted (Cost-Sensitive)": (LogisticRegression(class_weight='balanced'), X_raw, y_raw),
    "RUS Balanced": (LogisticRegression(), X_rus, y_rus),
    "SMOTE Balanced": (LogisticRegression(), X_smote, y_smote)
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Set mesh grid limits
x_min, x_max = X_raw[:, 0].min() - 1, X_raw[:, 0].max() + 1
y_min, y_max = X_raw[:, 1].min() - 1, X_raw[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

for i, (name, (model, X_train, y_train)) in enumerate(models.items()):
    model.fit(X_train, y_train)
    
    # Predict mesh grid output
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary contour
    ax = axes[i]
    ax.contourf(xx, yy, Z, alpha=0.15, cmap='coolwarm')
    
    # Plot original imbalanced scatter on top of the boundary to judge boundary placement
    ax.scatter(X_raw[y_raw == 0, 0], X_raw[y_raw == 0, 1], color='#3498db', alpha=0.5, label='Class 0')
    ax.scatter(X_raw[y_raw == 1, 0], X_raw[y_raw == 1, 1], color='#e74c3c', alpha=0.8, edgecolors='k', label='Class 1')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle("Logistic Regression Decision Boundaries under Different Balance Schemes", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 6. Demo C: Evaluation Comparison (ROC-AUC vs. PR-AUC)

We will now generate a highly skewed classification task ($10,000$ samples, **99% / 1% split**) and train a Random Forest Classifier. We'll observe how the ROC Curve and the Precision-Recall (PR) Curve reveal different aspects of performance.

In [ ]:
# Generate 10k highly imbalanced dataset (99% vs 1%)
X, y = make_classification(
    n_samples=10000, n_features=10, n_informative=8, n_classes=2,
    weights=[0.99, 0.01], random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Fit standard Random Forest Baseline
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

# Display Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Class 0 (Legit)', 'Class 1 (Fraud)']))

### Plotting ROC-AUC vs. PR-AUC
Let's plot both curves side by side and calculate the Area Under the Curve (AUC).

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Plot ROC Curve
axes[0].plot(fpr, tpr, color='#1abc9c', label=f'ROC Curve (AUC = {roc_auc:.4f})', linewidth=2.5)
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR / Recall)')
axes[0].set_title('ROC Curve\n(Looks highly optimistic due to massive TNs)', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right')

# Plot Precision-Recall Curve
axes[1].plot(recall, precision, color='#d35400', label=f'PR Curve (AUC = {pr_auc:.4f})', linewidth=2.5)
axes[1].axhline(y=np.sum(y_test==1)/len(y_test), color='red', linestyle='--', label=f'Random Guess Base Rate ({np.sum(y_test==1)/len(y_test):.2f})')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve\n(Reflects true capability of isolating minority class)', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

## Summary Checklist for Imbalanced Datasets

1. **Never Evaluate with Accuracy Alone:**
   - Always review **Precision**, **Recall**, and **F1-Score**.
   - Look at the **Precision-Recall Area Under the Curve (PR-AUC)** rather than ROC-AUC when positive instances are extremely rare.
2. **Resampling Techniques:**
   - Use **Random Under-sampling (RUS)** when you have excessive dataset sizes and need to cut compute footprint.
   - Use **SMOTE** or **ADASYN** to synthesize minority entries. Always perform resampling *only* on the training split—never touch your test set before evaluation!
3. **Algorithmic Solutions (Easiest to Start With):**
   - Apply cost-sensitive properties like `class_weight='balanced'` in logistic regression, trees, or SVM algorithms.
4. **Stratification (Crucial Workflow Step):**
   - When splitting datasets into train and test groups (`train_test_split`), always specify `stratify=y` to ensure consistent minority distributions in all splits.